# Lab 5: Customer Segmentation

**Challenge**: Group mall customers into segments based on their spending behavior.

**What you'll use**: pandas, scikit-learn, K-Means Clustering, visualization

---
### The Story
A shopping mall wants to understand their customers better. They have data on customer age, annual income, and spending score. Your job is to find natural customer groups so the marketing team can target each group differently!

In [ ]:
!pip install pandas scikit-learn matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print('✅ Ready!')

In [ ]:
# Create the Mall Customer dataset (based on the famous Kaggle dataset)
np.random.seed(42)
n = 200

# Simulate 5 customer groups
data = {
    'CustomerID': range(1, n+1),
    'Age': np.concatenate([
        np.random.normal(25, 5, 40),   # Young
        np.random.normal(45, 8, 40),   # Middle-aged
        np.random.normal(35, 6, 40),   # Mid-income
        np.random.normal(30, 5, 40),   # High earner young
        np.random.normal(50, 7, 40),   # Older low spender
    ]).clip(18, 70).astype(int),
    'Annual_Income_k': np.concatenate([
        np.random.normal(25, 5, 40),   # Low income
        np.random.normal(55, 10, 40),  # Mid income
        np.random.normal(55, 10, 40),  # Mid income
        np.random.normal(85, 10, 40),  # High income
        np.random.normal(85, 10, 40),  # High income
    ]).clip(15, 140).astype(int),
    'Spending_Score': np.concatenate([
        np.random.normal(50, 15, 40),  # Average spender
        np.random.normal(50, 15, 40),  # Average spender
        np.random.normal(80, 10, 40),  # High spender
        np.random.normal(80, 10, 40),  # High spender
        np.random.normal(20, 10, 40),  # Low spender
    ]).clip(1, 100).astype(int)
}

df = pd.DataFrame(data)
print('Dataset shape:', df.shape)
print(df.describe())

In [ ]:
# Explore the data
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df['Age'].hist(bins=20, ax=axes[0], color='#4ecdc4', alpha=0.8)
axes[0].set_title('Age Distribution')

df['Annual_Income_k'].hist(bins=20, ax=axes[1], color='#ff6b6b', alpha=0.8)
axes[1].set_title('Income Distribution ($k)')

df['Spending_Score'].hist(bins=20, ax=axes[2], color='#00ff9d', alpha=0.8)
axes[2].set_title('Spending Score (1-100)')

plt.tight_layout()
plt.show()

In [ ]:
# Use Income and Spending Score for clustering (most informative)
X = df[['Annual_Income_k', 'Spending_Score']].values

# YOUR TURN: Find the best number of clusters using the Elbow Method
# Try k from 1 to 10 and record the inertia (within-cluster sum of squares)

inertias = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertias, 'o-', color='#00ff9d', linewidth=2)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (lower = tighter clusters)')
plt.title('Elbow Method — Find the "elbow" point')
plt.xticks(k_range)
plt.show()
print('Look for the "elbow" — where the curve bends sharply. That is the best K!')

In [ ]:
# YOUR TURN: Based on the elbow plot, choose the best K
best_k = ___  # ← Set this (hint: look at the elbow, likely 5)

# Train the final K-Means model
kmeans = ___  # ← KMeans(n_clusters=best_k, random_state=42, n_init=10)
___  # ← Fit on X

df['Cluster'] = kmeans.labels_
print(f'Cluster sizes:')
print(df['Cluster'].value_counts().sort_index())

In [ ]:
# Visualize the clusters!
colors = ['#ff6b6b', '#4ecdc4', '#00ff9d', '#ffd93d', '#c77dff']
labels = ['Low Income\nLow Spend', 'Low Income\nHigh Spend', 'Mid Income\nMid Spend', 
          'High Income\nHigh Spend', 'High Income\nLow Spend']

plt.figure(figsize=(10, 6))
for i in range(best_k):
    cluster_data = df[df['Cluster'] == i]
    plt.scatter(cluster_data['Annual_Income_k'], cluster_data['Spending_Score'],
                color=colors[i % len(colors)], label=f'Cluster {i}', alpha=0.7, s=60)

# Plot centroids
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            marker='*', s=300, color='white', edgecolors='black', zorder=5, label='Centroids')

plt.xlabel('Annual Income ($k)')
plt.ylabel('Spending Score (1-100)')
plt.title('Customer Segments')
plt.legend()
plt.show()

In [ ]:
# ✅ TEST CELL
assert isinstance(best_k, (int, np.integer)), 'best_k must be an integer'
assert 3 <= best_k <= 7, 'best_k should be between 3 and 7 for this dataset'
assert hasattr(kmeans, 'labels_'), 'kmeans must be fitted'
assert len(set(kmeans.labels_)) == best_k, 'Number of unique clusters must equal best_k'
print('🎉 All tests passed!')
print(f'You found {best_k} customer segments!')
print('Marketing can now target each group differently.')